<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Visualización Temporal: Series de Tiempo y Datos a Nivel de Eventos 📅
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Visual Analytics and Critical Thinking
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 06 🧗
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Visual%20Analytics%20and%20Critical%20Thinking/06%20-%20Visualizacion%20Avanzada/01_Visualizacion_Temporal.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Esta subsección (2.4.2 del syllabus) cubre la **visualización temporal**: cómo representar datos que cambian a lo largo del tiempo, tanto en forma de **series continuas** (series de tiempo) como de **eventos discretos** (datos a nivel de evento).

1. Construir y graficar **series de tiempo** con `pandas.date_range`.
2. Aplicar **remuestreo** (`resample`) y medias móviles para suavizar y agregar series.
3. Descomponer una serie en **tendencia, estacionalidad y ruido**.
4. Visualizar **datos a nivel de eventos** (timestamps discretos) con gráficos de línea de tiempo (*event timelines*).
5. Comparar las herramientas de análisis temporal de Python con Power BI y Tableau.

> 🧗 **Nivel avanzado:** Este módulo asume dominio de los módulos 01-05.

---
## Recursos Recomendados 📚

- [Pandas Time Series Documentation](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [statsmodels — Seasonal Decompose](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html)
- [Plotly Time Series](https://plotly.com/python/time-series/)
- [Power BI — Time Intelligence (DAX)](https://learn.microsoft.com/es-es/dax/time-intelligence-functions-dax)
- [Tableau — Date & Time Functions](https://help.tableau.com/current/pro/desktop/es-es/functions_functions_date.htm)

---
## 1. Series de Tiempo: Construcción con `pandas.date_range` 📈

Una **serie de tiempo** es una secuencia de observaciones indexadas por tiempo, típicamente a intervalos regulares (diario, semanal, mensual). `pandas.date_range()` genera ese índice temporal de forma robusta (maneja fines de semana, años bisiestos, zonas horarias, etc.).

In [ ]:
# ============================================================
# Construcción de una serie de tiempo sintética con pandas
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Índice temporal: semanal, 3 años
fechas = pd.date_range('2022-01-01', '2024-12-31', freq='W')

tendencia = np.linspace(50_000, 78_000, len(fechas))
estacionalidad = 8_000 * np.sin(2 * np.pi * np.arange(len(fechas)) / 52)
ruido = np.random.normal(0, 2_000, len(fechas))
ventas = np.abs(tendencia + estacionalidad + ruido)

serie = pd.Series(ventas, index=fechas, name='ventas')
print(serie.head())
print(f"\nFrecuencia del índice: {serie.index.freqstr}")
print(f"Rango: {serie.index.min().date()} → {serie.index.max().date()}  ({len(serie)} observaciones)")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(serie.index, serie.values, color='#3b82f6', linewidth=1.3)
ax.fill_between(serie.index, serie.values, alpha=0.08, color='#3b82f6')
ax.set_title('Serie de Tiempo: Ventas Semanales (2022-2024)', fontweight='bold')
ax.set_ylabel('Ventas ($)')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## 2. Remuestreo (`resample`) y Medias Móviles 🔁

`resample()` cambia la frecuencia de una serie de tiempo (p.ej. de semanal a mensual), aplicando una función de agregación (`.mean()`, `.sum()`, `.max()`...). Es el equivalente temporal de un `groupby`.

Las **medias móviles** (`rolling`) suavizan el ruido de corto plazo para revelar la tendencia subyacente.

In [ ]:
# ============================================================
# Resample: cambiar la frecuencia de la serie + medias móviles
# ============================================================

# De semanal a mensual (promedio) y a trimestral (suma)
serie_mensual = serie.resample('ME').mean()
serie_trimestral = serie.resample('QE').sum()

print("Semanal → Mensual (promedio), primeras filas:")
print(serie_mensual.head())
print("\nSemanal → Trimestral (suma), primeras filas:")
print(serie_trimestral.head())

# Media móvil de 8 semanas sobre la serie original
media_movil = serie.rolling(window=8).mean()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)

axes[0].plot(serie.index, serie.values, color='#94a3b8', linewidth=1, label='Semanal (original)')
axes[0].plot(media_movil.index, media_movil.values, color='#dc2626', linewidth=2, label='Media móvil 8 sem')
axes[0].set_title('Suavizado con Media Móvil (rolling)', fontweight='bold')
axes[0].legend(); axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].bar(serie_mensual.index, serie_mensual.values, width=20, color='#10b981', alpha=0.85)
axes[1].set_title('Serie Remuestreada a Frecuencia Mensual (resample)', fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n📊 En Power BI: la granularidad se cambia arrastrando la jerarquía de Fecha (Año>Trimestre>Mes>Día).")
print("📈 En Tableau: clic derecho sobre el campo de fecha en Columns → elegir el nivel de detalle.")

---
## 3. Descomposición: Tendencia, Estacionalidad y Ruido 🧩

Toda serie de tiempo (modelo aditivo) puede descomponerse como:

$$\text{Serie} = \text{Tendencia} + \text{Estacionalidad} + \text{Residuo (ruido)}$$

`statsmodels.tsa.seasonal.seasonal_decompose` automatiza esta separación, muy útil para diagnosticar *por qué* una serie sube o baja.

In [ ]:
# ============================================================
# Descomposicion de series de tiempo con statsmodels (opcional)
# ============================================================

try:
    from statsmodels.tsa.seasonal import seasonal_decompose

    descomposicion = seasonal_decompose(serie, model='additive', period=52)

    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
    partes = [
        (serie, 'Serie Original', '#3b82f6'),
        (descomposicion.trend, 'Tendencia', '#10b981'),
        (descomposicion.seasonal, 'Estacionalidad (ciclo anual)', '#f59e0b'),
        (descomposicion.resid, 'Residuos (ruido)', '#ef4444'),
    ]
    for ax, (data, titulo, color) in zip(axes, partes):
        ax.plot(data.index, data.values, color=color, linewidth=1.3)
        ax.set_title(titulo, fontsize=10, fontweight='bold', loc='left')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    fig.suptitle('Descomposición de la Serie: Tendencia + Estacionalidad + Ruido', fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("ℹ️  statsmodels no está instalado (opcional). Instalar con: pip install statsmodels")

print("\n📊 En Power BI: no hay descomposición automática nativa; se aproxima con DAX")
print("   (medias móviles + SAMEPERIODLASTYEAR para aislar estacionalidad).")
print("📈 En Tableau: Analytics pane → Trend Line (tendencia). La estacionalidad se explora")
print("   manualmente comparando el mismo periodo entre años (small multiples).")

---
## 4. Datos a Nivel de Eventos (*Event-Level Data*) ⏱️

No todos los datos temporales son series continuas y regulares. Muchos son **eventos discretos**: transacciones, clics, tickets de soporte, fallas de un sistema — cada uno con un timestamp exacto, sin una frecuencia fija.

| Serie de tiempo | Datos a nivel de eventos |
|---|---|
| Un valor por intervalo regular (día, semana) | Un registro por ocurrencia, timestamp irregular |
| Se agrega naturalmente (`resample`) | Se agrega contando eventos en ventanas de tiempo |
| Se grafica como línea continua | Se grafica como línea de tiempo (*timeline*) o marcas discretas |
| Ejemplo: ventas diarias totales | Ejemplo: cada transacción individual con su hora exacta |

Para visualizarlos usamos un **gráfico de línea de tiempo** (*event timeline* / *swimlane*): cada evento es un punto o marca ubicado en el eje del tiempo, con su categoría en el eje Y.

In [ ]:
# ============================================================
# Datos a nivel de eventos: generación y timeline (swimlane)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

np.random.seed(11)

categorias_evento = ['Login', 'Compra', 'Error 500', 'Soporte Ticket', 'Cancelación']
n_eventos = 120

# Timestamps irregulares dentro de un mes (a diferencia del resample regular)
inicio = pd.Timestamp('2024-06-01')
offsets_minutos = np.sort(np.random.randint(0, 30*24*60, n_eventos))
timestamps = [inicio + pd.Timedelta(minutes=int(m)) for m in offsets_minutos]
categorias = np.random.choice(categorias_evento, n_eventos,
                               p=[0.35, 0.25, 0.10, 0.20, 0.10])

eventos = pd.DataFrame({'timestamp': timestamps, 'tipo_evento': categorias})
print(eventos.head())
print(f"\nTotal de eventos: {len(eventos)}  |  Tipos: {eventos['tipo_evento'].nunique()}")

# --- Gráfico de línea de tiempo (timeline / swimlane) ---
fig, ax = plt.subplots(figsize=(11, 4.5))
colores = dict(zip(categorias_evento, ['#3b82f6', '#10b981', '#ef4444', '#f59e0b', '#8b5cf6']))

for i, cat in enumerate(categorias_evento):
    sub = eventos[eventos['tipo_evento'] == cat]
    ax.scatter(sub['timestamp'], [i] * len(sub), color=colores[cat], s=35, alpha=0.75, label=cat)

ax.set_yticks(range(len(categorias_evento)))
ax.set_yticklabels(categorias_evento)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
ax.set_title('Línea de Tiempo de Eventos (Event Timeline) — Junio 2024', fontweight='bold')
ax.set_xlabel('Fecha')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

# --- Agregación de eventos discretos en una serie regular (conteo diario) ---
eventos_por_dia = eventos.set_index('timestamp').resample('D').size()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(eventos_por_dia.index, eventos_por_dia.values, color='#1e3a8a', width=0.8)
ax.set_title('Eventos discretos agregados a serie regular (conteo diario)', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("\n💡 Convertir eventos discretos en serie regular = agrupar con resample('D').size()")
print("   Este es el puente entre 'datos a nivel de evento' y 'series de tiempo' clásicas.")

---
## 5. Herramientas para Dashboards Temporales: Python vs BI 🔺

| Capacidad | Python (pandas/plotly) | Power BI | Tableau |
|---|---|---|---|
| **Remuestreo/agregación** | `.resample()` | Jerarquía de fecha automática | Nivel de detalle en Columns |
| **Medias móviles** | `.rolling().mean()` | Medida DAX con `AVERAGEX` + rango | Analytics → Moving Average |
| **Comparación interanual (YoY)** | `.shift(52)` o merge por año | `SAMEPERIODLASTYEAR()` (DAX) | Cálculo de tabla (Table Calc) |
| **Eventos discretos / timeline** | `scatter` sobre timestamps | Visual de línea de tiempo (Gantt) | Gantt Chart |
| **Descomposición estacional** | `statsmodels.seasonal_decompose` | No nativo | No nativo |

> 🚀 **Siguiente cuaderno:** [02_Visualizacion_Multivariada.ipynb](02_Visualizacion_Multivariada.ipynb) — Relaciones entre múltiples variables.

---
## Resumen y Puntos Clave 🎯

- `pandas.date_range()` construye índices temporales robustos; una **serie de tiempo** es una secuencia de valores indexada por tiempo.
- `resample()` cambia la frecuencia de la serie agregando (media, suma, etc.); `rolling()` suaviza con medias móviles.
- La **descomposición** separa una serie en tendencia + estacionalidad + ruido, ayudando a diagnosticar patrones.
- Los **datos a nivel de eventos** tienen timestamps irregulares y se visualizan con líneas de tiempo (*timelines*); se pueden agregar a una serie regular con `resample(...).size()`.
- Power BI y Tableau ofrecen jerarquías de fecha e inteligencia de tiempo integradas sin código, pero la descomposición estacional formal requiere Python/R.

### 🧠 Autoevaluación

1. ¿Cuál es la diferencia conceptual entre una serie de tiempo regular y datos a nivel de eventos? Da un ejemplo de cada una que no esté en el cuaderno.
2. ¿Qué hace exactamente `serie.resample('ME').mean()`? ¿En qué se diferencia de `serie.rolling(4).mean()`?
3. En la descomposición aditiva (Serie = Tendencia + Estacionalidad + Residuo), ¿qué representaría un residuo con un pico muy grande en una fecha específica?
4. ¿Cómo convertirías una tabla de eventos discretos (timestamp + tipo) en una serie de tiempo diaria contable? Escribe el patrón de código (no hace falta ejecutarlo).